# Basic Usage Guide

This notebook covers the fundamentals of the simple-backtest framework:

1. Loading data with yfinance
2. Setting up commissions
3. Running a backtest with Moving Average strategy
4. Comparing multiple strategies (Buy and Hold, DCA, Moving Average)
5. Simple parameter optimization

Let's get started!

In [ ]:
# Install the library (run this cell if using Colab or if you haven't installed the package)
!pip install --upgrade --no-cache-dir simple-backtest yfinance

In [ ]:
import yfinance as yf

from simple_backtest import Backtest, BacktestConfig
from simple_backtest.optimization import GridSearchOptimizer
from simple_backtest.strategy import BuyAndHoldStrategy, DCAStrategy, MovingAverageStrategy
from simple_backtest.visualization import plot_equity_curve

## 1. Loading Data with yfinance

We'll download historical stock data for Apple (AAPL) using yfinance. The framework requires OHLCV data with a DatetimeIndex.

In [ ]:
# Download data from Yahoo Finance
ticker = "AAPL"
start_date = "2020-01-01"
end_date = "2023-12-31"

print(f"Downloading {ticker} data from {start_date} to {end_date}...")
data = yf.download(ticker, start=start_date, end=end_date, progress=False)

# Display first few rows
print(f"\nData shape: {data.shape}")
print(f"Date range: {data.index[0]} to {data.index[-1]}")
data.head()

In [ ]:
# Check for missing values
print("Missing values:")
print(data.isnull().sum())

# Drop any rows with missing values
data = data.dropna()
print(f"\nClean data shape: {data.shape}")

## 2. Setting Up Commission

The framework supports multiple commission types:
- **Percentage**: Commission as percentage of trade value (e.g., 0.1%)
- **Flat**: Fixed commission per trade (e.g., $1 per trade)
- **Tiered**: Different rates for different trade sizes

Let's start with a simple percentage commission.

In [ ]:
# Configure backtest with percentage commission
config = BacktestConfig(
    initial_capital=10000.0,  # Start with $10,000
    lookback_period=50,  # Use 50 days of historical data
    commission_type="percentage",  # Percentage-based commission
    commission_value=0.001,  # 0.1% per trade
    execution_price="open",  # Execute at open price
    risk_free_rate=0.02,  # 2% annual risk-free rate
)

print("Backtest Configuration:")
print(f"  Initial Capital: ${config.initial_capital:,.2f}")
print(f"  Commission: {config.commission_value * 100}% per trade")
print(f"  Lookback Period: {config.lookback_period} days")

## 3. Running a Backtest with Moving Average Strategy

The Moving Average Crossover strategy:
- **Buy** when short MA crosses above long MA (golden cross)
- **Sell** when short MA crosses below long MA (death cross)

In [ ]:
# Create Moving Average strategy
ma_strategy = MovingAverageStrategy(
    short_window=10,  # 10-day moving average
    long_window=30,  # 30-day moving average
    shares=10,  # Trade 10 shares at a time
)

print(f"Strategy: {ma_strategy.get_name()}")
print(f"  Short Window: {ma_strategy.short_window} days")
print(f"  Long Window: {ma_strategy.long_window} days")
print(f"  Shares per trade: {ma_strategy.shares}")

In [ ]:
# Run the backtest
backtest = Backtest(data=data, config=config)
results = backtest.run([ma_strategy])

print("Backtest completed!\n")

In [ ]:
# Display performance metrics
ma_result = results.get_strategy(ma_strategy.get_name())
metrics = ma_result.metrics

print("=" * 60)
print(f"Performance Metrics - {ma_strategy.get_name()}")
print("=" * 60)
print("\nReturns:")
print(f"  Total Return: {metrics['total_return']:.2f}%")
print(f"  CAGR: {metrics['cagr']:.2f}%")
print("\nRisk Metrics:")
print(f"  Sharpe Ratio: {metrics['sharpe_ratio']:.2f}")
print(f"  Sortino Ratio: {metrics['sortino_ratio']:.2f}")
print(f"  Max Drawdown: {metrics['max_drawdown']:.2f}%")
print(f"  Volatility: {metrics['volatility']:.2f}%")
print("\nTrade Statistics:")
print(f"  Total Trades: {int(metrics['total_trades'])}")
print(f"  Win Rate: {metrics['win_rate']:.2f}%")
print(f"  Profit Factor: {metrics['profit_factor']:.2f}")
print("\nFinal Values:")
print(f"  Final Portfolio Value: ${metrics['final_value']:,.2f}")
print(f"  Initial Capital: ${config.initial_capital:,.2f}")
print("=" * 60)

In [ ]:
# Visualize trading signals with buy/sell markers
fig = ma_result.plot_trades(data)
fig.show()

## 4. Comparing Multiple Strategies

Let's compare three different strategies:
1. **Moving Average Crossover** - Active trading strategy
2. **Buy and Hold** - Passive benchmark strategy
3. **Dollar Cost Averaging (DCA)** - Regular investment strategy

In [ ]:
# Create multiple strategies
strategies = [
    MovingAverageStrategy(short_window=10, long_window=30, shares=10, name="MA_10_30"),
    BuyAndHoldStrategy(
        shares=50,  # Buy 50 shares at the start
        name="BuyAndHold",
    ),
    DCAStrategy(
        investment_amount=500,  # Invest $500 each time
        interval_days=30,  # Every 30 days
        name="DCA_Monthly",
    ),
]

print("Strategies to compare:")
for s in strategies:
    print(f"  - {s.get_name()}")

In [ ]:
# Run backtest with all strategies
backtest = Backtest(data=data, config=config)
results = backtest.run(strategies)

print("Backtest completed for all strategies!\n")

In [ ]:
# Compare results - specify metrics to include total_trades
comparison_df = results.compare(
    metrics=["total_return", "cagr", "sharpe_ratio", "max_drawdown", "total_trades", "win_rate"]
)
print("\nStrategy Comparison:")
print(comparison_df)

In [ ]:
# Visualize trading signals for all strategies
trade_figures = results.plot_trades(data)
for name, fig in trade_figures.items():
    fig.show()

In [ ]:
# Find best strategy by Sharpe ratio
best_result = results.best_strategy("sharpe_ratio")
print(f"\nBest Strategy (by Sharpe Ratio): {best_result.name}")
print(f"Sharpe Ratio: {best_result.metrics['sharpe_ratio']:.2f}")

In [ ]:
# Visualize comparison
fig = results.plot_comparison()
fig.update_layout(title="Strategy Comparison - Equity Curves")
fig.show()

## 5. Parameter Optimization

Let's optimize the Moving Average strategy by testing different window combinations to find the best parameters.

In [ ]:
# Define parameter space to search
param_space = {
    "short_window": [5, 10, 15, 20],  # Test different short windows
    "long_window": [30, 40, 50, 60],  # Test different long windows
    "shares": [10],  # Keep shares constant
}

print("Parameter Space:")
for param, values in param_space.items():
    print(f"  {param}: {values}")

# Calculate total combinations
total = 1
for values in param_space.values():
    total *= len(values)
print(f"\nTotal combinations to test: {total}")

In [ ]:
# Run optimization
optimizer = GridSearchOptimizer(verbose=True)

optimization_results = optimizer.optimize(
    data=data,
    config=config,
    strategy_class=MovingAverageStrategy,
    param_space=param_space,
    metric="sharpe_ratio",  # Optimize for Sharpe ratio
)

print("\nOptimization completed!")

In [ ]:
# Display top 10 parameter combinations
print("\nTop 10 Parameter Combinations (by Sharpe Ratio):")
print("=" * 80)
top_10 = optimization_results.head(10)[
    ["short_window", "long_window", "sharpe_ratio", "total_return", "max_drawdown", "total_trades"]
]
print(top_10.to_string(index=False))

In [ ]:
# Get best parameters
best_params = optimization_results.iloc[0]

print("\nBest Parameters:")
print(f"  Short Window: {int(best_params['short_window'])} days")
print(f"  Long Window: {int(best_params['long_window'])} days")
print("\nPerformance with Best Parameters:")
print(f"  Sharpe Ratio: {best_params['sharpe_ratio']:.2f}")
print(f"  Total Return: {best_params['total_return']:.2f}%")
print(f"  Max Drawdown: {best_params['max_drawdown']:.2f}%")
print(f"  Total Trades: {int(best_params['total_trades'])}")

In [ ]:
# Run backtest with optimized parameters
optimized_strategy = MovingAverageStrategy(
    short_window=int(best_params["short_window"]),
    long_window=int(best_params["long_window"]),
    shares=10,
    name="MA_Optimized",
)

backtest = Backtest(data=data, config=config)
optimized_results = backtest.run([optimized_strategy])

# Visualize
fig = plot_equity_curve(optimized_results)
fig.update_layout(title="Optimized Moving Average Strategy")
fig.show()

In [ ]:
# Visualize trading signals for optimized strategy
optimized_result = optimized_results.get_strategy("MA_Optimized")
fig = optimized_result.plot_trades(data)
fig.show()

## Summary

In this notebook, we covered:

1. ✅ **Data Loading**: Using yfinance to download OHLCV data
2. ✅ **Commission Setup**: Configuring percentage-based commissions
3. ✅ **Running Backtests**: Testing a Moving Average strategy
4. ✅ **Strategy Comparison**: Comparing MA, Buy & Hold, and DCA strategies
5. ✅ **Parameter Optimization**: Using GridSearch to find optimal MA parameters

### Key Takeaways:

- The framework is **asset-agnostic** - works with stocks, forex, crypto, etc.
- Multiple commission types are supported (percentage, flat, tiered)
- Strategies can be easily compared using built-in comparison methods
- Parameter optimization helps find the best strategy configuration
- The framework provides 20+ performance metrics automatically

### Next Steps:

- Explore candlestick pattern strategies
- Learn technical indicator strategies (RSI, MACD, Bollinger Bands)
- Create machine learning-based strategies
- Implement custom commission models
- Use advanced optimization techniques